# 5. User Profiling & Personalized Recommendations

Turn one user's listening history into a taste vector, then use it to search the CLAP index for recommendations.

```
recording_play (Postgres: userId, recordingId, createdAt)
        │
        ▼  filter to USER_ID
  recency weight  =  0.5 ** (age_days / HALF_LIFE)
        │
        ▼
   join to per-recording features already built by notebooks 2-4
        │
   ┌────┼──────────────┬───────────────────┐
   ▼    ▼              ▼                   ▼
 CLAP  YAMNet class   transcript          (is_speech, filename)
 512-D distribution    768-D
   │    │              │
   ▼    ▼              ▼
 weighted-average, re-normalized  →  one user's profile
        │
        ▼
FAISS search (CLAP) for candidates, reranked by blending
CLAP similarity + YAMNet overlap + transcript similarity
        │
        ▼
   top-K personalized recommendations
```

**Requires notebook 2 (CLAP_embedding_and_Indexing) to have been run through its save step**, so `data_index/clap_embeddings.faiss` and `data_index/clap_metadata.json` exist -- this notebook queries that index directly rather than rebuilding it.

The notebook is scoped to **one user at a time** (set `USER_ID` below). Building their profile only touches that user's plays, and saving it *upserts* into `data/user_profiles.csv` -- any other users already cached there from a previous run are left untouched. Rerun with a different `USER_ID` to add another user to the cache.

Users with no play history (cold start) fall back to a popularity ranking instead of a profile.


## 0. Imports and configuration

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# --- Database (same credentials/table style as notebook 1) ---
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Notebook 1 reads `recording` (content). This notebook reads `recording_play`
# (who played what, when) -- a separate table, so it gets its own env vars
# rather than overloading notebook 1's TABLE_NAME/WHERE_CLAUSE/LIMIT.
PLAYS_TABLE = os.getenv("PLAYS_TABLE", "public.recording_play")
_plays_where_env = os.getenv("PLAYS_WHERE_CLAUSE", "").strip()
PLAYS_WHERE_CLAUSE = _plays_where_env if _plays_where_env else None
_plays_limit_env = os.getenv("PLAYS_LIMIT", "").strip()
PLAYS_LIMIT = int(_plays_limit_env) if _plays_limit_env else None

# Re-query the DB even if a cached recording_plays.csv already exists.
FORCE_REFRESH_PLAYS = os.getenv("FORCE_REFRESH_PLAYS", "").strip().lower() in {"1", "true", "yes"}

# --- Paths (relative to this notebook's directory, i.e. jupyter_notebooks/) ---
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
INDEX_DIR = PROJECT_ROOT / "data_index"

RECORDING_PLAYS_CSV = DATA_DIR / os.getenv("RECORDING_PLAYS_CSV", "recording_plays.csv")

CLAP_INDEX_PATH = INDEX_DIR / "clap_embeddings.faiss"
CLAP_META_PATH = INDEX_DIR / "clap_metadata.json"
YAMNET_CSV = DATA_DIR / "yamnet_top5_classifications.csv"
TRANSCRIPTS_CSV = DATA_DIR / "audio_transcriptions.csv"
TRANSCRIPT_EMBEDDINGS_CSV = DATA_DIR / "audio_transcript_embeddings.csv"

USER_PROFILES_CSV = DATA_DIR / "user_profiles.csv"
YAMNET_VOCAB_JSON = INDEX_DIR / "yamnet_vocab.json"

ID_COLUMN = "id"

# --- Profiling knobs ---
# Half-life for recency weighting: a play this many days old counts for
# half as much as one made today. Recent listens dominate the taste vector,
# but old ones never drop to exactly zero.
RECENCY_HALF_LIFE_DAYS = float(os.getenv("RECENCY_HALF_LIFE_DAYS", "30"))

# --- Recommendation knobs ---
TOP_K = int(os.getenv("TOP_K", "10"))
CANDIDATE_POOL = int(os.getenv("CANDIDATE_POOL", "200"))  # FAISS candidates before reranking

# How much each signal contributes to the final rerank score. Redistributed
# per-candidate across whichever signals are actually available (e.g. a
# non-speech candidate has no transcript_sim, so its weight goes to the rest).
W_CLAP = float(os.getenv("W_CLAP", "0.60"))
W_YAMNET = float(os.getenv("W_YAMNET", "0.15"))
W_TRANSCRIPT = float(os.getenv("W_TRANSCRIPT", "0.25"))

DATA_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f"recency half-life: {RECENCY_HALF_LIFE_DAYS} days")
print(f"rerank weights: clap={W_CLAP} yamnet={W_YAMNET} transcript={W_TRANSCRIPT}")


## 1. Stage 0 — Export listening history from Postgres

Same cache-once pattern as notebook 1: reused as-is unless `FORCE_REFRESH_PLAYS=1`. All users' plays are pulled
here (it's one cheap query) -- the per-user scoping happens later, in Section 4.


In [ ]:
def connect_to_database():
    return psycopg.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
    )


def load_play_events() -> pd.DataFrame:
    query = f"SELECT * FROM {PLAYS_TABLE}"
    if PLAYS_WHERE_CLAUSE:
        query += f" WHERE {PLAYS_WHERE_CLAUSE}"
    if PLAYS_LIMIT is not None:
        query += f" LIMIT {int(PLAYS_LIMIT)}"

    print("Reading play events from PostgreSQL...")
    with connect_to_database() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            column_names = [description.name for description in cursor.description]
            rows = cursor.fetchall()

    dataframe = pd.DataFrame(rows, columns=column_names)
    print(f"Loaded {len(dataframe)} play events.")
    return dataframe


In [ ]:
if RECORDING_PLAYS_CSV.exists() and not FORCE_REFRESH_PLAYS:
    plays_df = pd.read_csv(RECORDING_PLAYS_CSV)
    print(
        f"Loaded {len(plays_df)} cached rows from {RECORDING_PLAYS_CSV}.\n"
        "Set FORCE_REFRESH_PLAYS=1 in .env to re-query the DB instead."
    )
else:
    plays_df = load_play_events()
    plays_df.to_csv(RECORDING_PLAYS_CSV, index=False)
    print(f"Saved {len(plays_df)} rows to {RECORDING_PLAYS_CSV}")

plays_df["userId"] = plays_df["userId"].astype(str)
plays_df["recordingId"] = plays_df["recordingId"].astype(str)
plays_df["createdAt"] = pd.to_datetime(plays_df["createdAt"])

print(f"\ndistinct users:      {plays_df['userId'].nunique()}")
print(f"distinct recordings: {plays_df['recordingId'].nunique()}")


## 2. Load per-recording features built by notebooks 2-4

Three independent feature sources, each keyed by recording id, each with its own `ids` / `matrix` / `id_to_idx`
so building a weighted average is just indexing into a matrix.


In [ ]:
def l2_normalize(vector: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 0 else vector


def decode_vector(text: str) -> np.ndarray:
    return np.array(text.split(","), dtype=np.float32)


In [ ]:
# --- CLAP audio embeddings (notebook 2) — this is also the FAISS index we search ---
if not CLAP_INDEX_PATH.exists() or not CLAP_META_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CLAP_INDEX_PATH.name} / {CLAP_META_PATH.name} in {INDEX_DIR}.\n"
        "Run '2. CLAP_embedding_and_Indexing.ipynb' through its save step first."
    )

clap_index = faiss.read_index(str(CLAP_INDEX_PATH))
clap_metadata = json.loads(CLAP_META_PATH.read_text())

clap_ids = [str(m["id"]) for m in clap_metadata]
id_to_clap_idx = {rid: i for i, rid in enumerate(clap_ids)}
clap_matrix = clap_index.reconstruct_n(0, clap_index.ntotal).astype(np.float32)

is_speech_map = {str(m["id"]): bool(m["is_speech"]) for m in clap_metadata}
filename_map = {str(m["id"]): m["filename"] for m in clap_metadata}

print(f"CLAP index: {clap_index.ntotal} recordings, {clap_matrix.shape[1]}-D")


In [ ]:
# --- YAMNet class distribution (notebook: YAMNet_Top5) ---
# Melt the 5 rank columns into long (id, class_name, score) rows, then pivot
# into a dense [recording x class] matrix, normalized to a distribution.
yamnet_df = pd.read_csv(YAMNET_CSV)
yamnet_df[ID_COLUMN] = yamnet_df[ID_COLUMN].astype(str)
if "yamnet_status" in yamnet_df.columns:
    yamnet_df = yamnet_df[yamnet_df["yamnet_status"] == "completed"]

long_rows = []
for i in range(1, 6):
    sub = yamnet_df[[ID_COLUMN, f"yamnet_rank_{i}_class", f"yamnet_rank_{i}_mean_score"]].dropna()
    sub.columns = [ID_COLUMN, "class_name", "score"]
    long_rows.append(sub)
yamnet_long = pd.concat(long_rows, ignore_index=True)
yamnet_long = yamnet_long.groupby([ID_COLUMN, "class_name"], as_index=False)["score"].sum()

yamnet_vocab = sorted(yamnet_long["class_name"].unique())
class_to_col = {c: i for i, c in enumerate(yamnet_vocab)}

yamnet_ids = sorted(yamnet_long[ID_COLUMN].unique())
id_to_yamnet_idx = {rid: i for i, rid in enumerate(yamnet_ids)}

yamnet_matrix = np.zeros((len(yamnet_ids), len(yamnet_vocab)), dtype=np.float32)
for row in yamnet_long.itertuples(index=False):
    yamnet_matrix[id_to_yamnet_idx[getattr(row, ID_COLUMN)], class_to_col[row.class_name]] = row.score

row_sums = yamnet_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
yamnet_matrix /= row_sums  # each row is now a probability-like distribution over classes

print(f"YAMNet: {len(yamnet_ids)} recordings, {len(yamnet_vocab)} distinct classes")


In [ ]:
# --- Transcript embeddings (notebook 4) — speech recordings only ---
transcript_emb_df = pd.read_csv(TRANSCRIPT_EMBEDDINGS_CSV)
transcript_emb_df[ID_COLUMN] = transcript_emb_df[ID_COLUMN].astype(str)

transcript_ids = transcript_emb_df[ID_COLUMN].tolist()
id_to_transcript_idx = {rid: i for i, rid in enumerate(transcript_ids)}
transcript_matrix = np.vstack(
    transcript_emb_df["transcript_embedding"].apply(decode_vector).values
).astype(np.float32)

transcripts_df = pd.read_csv(TRANSCRIPTS_CSV)[[ID_COLUMN, "transcript"]]
transcripts_df[ID_COLUMN] = transcripts_df[ID_COLUMN].astype(str)
transcript_text_map = dict(zip(transcripts_df[ID_COLUMN], transcripts_df["transcript"]))

print(f"Transcript embeddings: {len(transcript_ids)} recordings, {transcript_matrix.shape[1]}-D")


## 3. Attach recency weights and content features to each play event

A play only contributes to a profile once its recording has a CLAP embedding (the FAISS index defines what's
recommendable). YAMNet/transcript features are attached when available but aren't required. Done across all
plays once, up front, so both the per-user profile below and the global popularity fallback can reuse it.


In [ ]:
plays_df["clap_idx"] = plays_df["recordingId"].map(id_to_clap_idx)

before = len(plays_df)
plays_df = plays_df.dropna(subset=["clap_idx"]).copy()
plays_df["clap_idx"] = plays_df["clap_idx"].astype(int)
after = len(plays_df)

print(f"Plays with a CLAP embedding: {after} / {before} ({after / before:.1%})")
print("(the rest reference recordings notebook 2 hasn't embedded yet)")

plays_df["yamnet_idx"] = plays_df["recordingId"].map(id_to_yamnet_idx)
plays_df["yamnet_idx"] = plays_df["yamnet_idx"].fillna(-1).astype(int)

plays_df["transcript_idx"] = plays_df["recordingId"].map(id_to_transcript_idx)
plays_df["transcript_idx"] = plays_df["transcript_idx"].fillna(-1).astype(int)

plays_df["is_speech"] = plays_df["recordingId"].map(is_speech_map)

# Recency weight: a play this many days old counts for half as much as one made today.
NOW = datetime.utcnow()
age_days = (NOW - plays_df["createdAt"]).dt.total_seconds() / 86400
age_days = age_days.clip(lower=0)  # guard against clock skew producing negative ages
plays_df["recency_weight"] = 0.5 ** (age_days / RECENCY_HALF_LIFE_DAYS)

plays_df[["userId", "recordingId", "createdAt", "recency_weight", "is_speech"]].head()


## 4. Pick a user

Set `USER_ID` to whichever user you want a profile/recommendations for -- any `userId` from
`data/recording_plays.csv` works. Defaults to the most active user in this dataset as a working example.


In [ ]:
USER_ID = plays_df.groupby("userId").size().idxmax()  # <- replace with any userId to check someone else

print(f"USER_ID = {USER_ID!r}")


## 5. Build this user's profile

Recency-weighted average of CLAP vectors (renormalized -- the primary signal, used to query FAISS),
recency-weighted average YAMNet distribution (interpretable "taste breakdown"), and recency-weighted average
transcript embedding over their speech plays only (`None` if they've never played speech). Only `USER_ID`'s
plays are touched -- no other user's data is read here.


In [ ]:
def build_profile_for_user(user_id: str, plays: pd.DataFrame) -> dict:
    group = plays[plays["userId"] == user_id]
    if group.empty:
        raise ValueError(f"No plays with a CLAP embedding found for user {user_id!r}.")

    weights = group["recency_weight"].to_numpy()

    clap_vec = l2_normalize(np.average(clap_matrix[group["clap_idx"].to_numpy()], axis=0, weights=weights))

    yamnet_mask = group["yamnet_idx"].to_numpy() >= 0
    if yamnet_mask.any():
        rows_ = yamnet_matrix[group.loc[yamnet_mask, "yamnet_idx"].to_numpy()]
        yamnet_vec = np.average(rows_, axis=0, weights=weights[yamnet_mask])
        total = yamnet_vec.sum()
        yamnet_vec = yamnet_vec / total if total > 0 else None
    else:
        yamnet_vec = None

    transcript_mask = group["transcript_idx"].to_numpy() >= 0
    if transcript_mask.any():
        rows_ = transcript_matrix[group.loc[transcript_mask, "transcript_idx"].to_numpy()]
        transcript_vec = l2_normalize(np.average(rows_, axis=0, weights=weights[transcript_mask]))
    else:
        transcript_vec = None

    return {
        "userId": user_id,
        "total_plays": len(group),
        "distinct_recordings": group["recordingId"].nunique(),
        "last_active": group["createdAt"].max(),
        "speech_play_ratio": float(group["is_speech"].mean()),
        "clap_profile": clap_vec,
        "yamnet_profile": yamnet_vec,
        "transcript_profile": transcript_vec,
    }


profile = build_profile_for_user(USER_ID, plays_df)

print(f"user:                {profile['userId']}")
print(f"total plays:         {profile['total_plays']}")
print(f"distinct recordings: {profile['distinct_recordings']}")
print(f"speech play ratio:   {profile['speech_play_ratio']:.1%}")
print(f"last active:         {profile['last_active']}")


## 6. Save (upsert) this user's profile

Vectors are stored comma-encoded, matching the convention already used for `audio_transcript_embeddings.csv`.
Any existing row for `USER_ID` in `user_profiles.csv` is replaced; every other cached user's row is left as-is.


In [ ]:
def encode_vector(vector) -> str:
    if vector is None:
        return ""
    return ",".join(f"{value:.6f}" for value in vector)


new_row = pd.DataFrame([{
    "userId": profile["userId"],
    "total_plays": profile["total_plays"],
    "distinct_recordings": profile["distinct_recordings"],
    "last_active": profile["last_active"],
    "speech_play_ratio": profile["speech_play_ratio"],
    "clap_profile": encode_vector(profile["clap_profile"]),
    "yamnet_profile": encode_vector(profile["yamnet_profile"]),
    "transcript_profile": encode_vector(profile["transcript_profile"]),
}])

if USER_PROFILES_CSV.exists():
    existing = pd.read_csv(USER_PROFILES_CSV)
    existing = existing[existing["userId"] != USER_ID]  # drop the stale row for this user, if any
    updated = pd.concat([existing, new_row], ignore_index=True)
else:
    updated = new_row

updated.to_csv(USER_PROFILES_CSV, index=False)
YAMNET_VOCAB_JSON.write_text(json.dumps(yamnet_vocab, indent=2))

print(f"upserted {USER_ID} into {USER_PROFILES_CSV} ({len(updated)} user(s) cached total)")
print(f"saved {YAMNET_VOCAB_JSON}")


## 7. Load cached profiles

`recommend_for_user()` below looks users up here rather than taking `profile` directly, so it works for
*any* user already cached in `user_profiles.csv` -- not just the one just built.


In [ ]:
def decode_vector_or_none(text):
    if pd.isna(text) or text == "":
        return None
    return decode_vector(text)


cached_profiles = pd.read_csv(USER_PROFILES_CSV)
cached_profiles["userId"] = cached_profiles["userId"].astype(str)
for col in ["clap_profile", "yamnet_profile", "transcript_profile"]:
    cached_profiles[col] = cached_profiles[col].apply(decode_vector_or_none)

user_profiles = cached_profiles.set_index("userId")
print(f"{len(user_profiles)} cached user profile(s): {list(user_profiles.index)}")


## 8. Popularity fallback (cold start)

Users with no plays -- or not yet cached in `user_profiles.csv` -- get the most-played recordings instead of a
personalized ranking. This is a global, catalog-level stat, so it's computed over all users' plays regardless
of `USER_ID`.


In [ ]:
popularity = (
    plays_df.groupby("recordingId")
    .size()
    .sort_values(ascending=False)
    .rename("play_count")
    .reset_index()
)
popularity["filename"] = popularity["recordingId"].map(filename_map)
popularity["is_speech"] = popularity["recordingId"].map(is_speech_map)

popularity.head(TOP_K)


## 9. Recommend

Retrieve a broad candidate pool from FAISS by CLAP similarity, drop what the user's already played, then rerank
by blending in YAMNet overlap and transcript similarity wherever those signals exist for a given candidate.


In [ ]:
def blended_score(clap_sim, yamnet_overlap, transcript_sim) -> float:
    """Weighted blend, redistributing weight across whichever signals are actually available."""
    components = [(clap_sim, W_CLAP)]
    if yamnet_overlap is not None:
        components.append((yamnet_overlap, W_YAMNET))
    if transcript_sim is not None:
        components.append((transcript_sim, W_TRANSCRIPT))

    total_weight = sum(w for _, w in components)
    return sum(v * w for v, w in components) / total_weight


def recommend_for_user(user_id: str, k: int = TOP_K) -> pd.DataFrame:
    if user_id not in user_profiles.index:
        print(f"No cached profile for {user_id} -- falling back to popularity.")
        return popularity.head(k).assign(reason="popularity_fallback")

    user_profile = user_profiles.loc[user_id]
    already_played = set(plays_df.loc[plays_df["userId"] == user_id, "recordingId"])

    pool = min(CANDIDATE_POOL, clap_index.ntotal)
    scores, idxs = clap_index.search(user_profile["clap_profile"].reshape(1, -1).astype("float32"), pool)

    user_yamnet = user_profile["yamnet_profile"]
    user_transcript = user_profile["transcript_profile"]

    results = []
    for clap_sim, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        rid = clap_ids[idx]
        if rid in already_played:
            continue

        yamnet_overlap = None
        if user_yamnet is not None and rid in id_to_yamnet_idx:
            yamnet_overlap = float(np.dot(user_yamnet, yamnet_matrix[id_to_yamnet_idx[rid]]))

        transcript_sim = None
        if user_transcript is not None and rid in id_to_transcript_idx:
            transcript_sim = float(np.dot(user_transcript, transcript_matrix[id_to_transcript_idx[rid]]))

        results.append({
            "recordingId": rid,
            "filename": filename_map.get(rid),
            "is_speech": is_speech_map.get(rid),
            "clap_sim": float(clap_sim),
            "yamnet_overlap": yamnet_overlap,
            "transcript_sim": transcript_sim,
            "score": blended_score(float(clap_sim), yamnet_overlap, transcript_sim),
        })

    ranked = pd.DataFrame(results).sort_values("score", ascending=False).head(k).reset_index(drop=True)
    ranked["transcript_snippet"] = ranked["recordingId"].map(
        lambda rid: (transcript_text_map.get(rid, "") or "")[:80]
    )
    return ranked


## 10. Demo — this user

In [ ]:
target_profile = user_profiles.loc[USER_ID]

if target_profile["yamnet_profile"] is not None:
    top_classes = sorted(zip(yamnet_vocab, target_profile["yamnet_profile"]), key=lambda x: -x[1])[:5]
    print("top YAMNet classes:")
    for class_name, weight in top_classes:
        print(f"  {weight:.3f}  {class_name}")


In [ ]:
recommend_for_user(USER_ID)


### Cold-start example

In [ ]:
recommend_for_user("00000000-0000-0000-0000-000000000000")


## 11. Play the recommended audio

Renders an inline audio player for each recommended clip, straight from notebook 1's `data/audio/` cache
(the same files `filename_map` already points at -- nothing is re-downloaded).


In [ ]:
from IPython.display import Audio, display

AUDIO_DIR = DATA_DIR / "audio"


def play_recommendations(recommendations: pd.DataFrame, max_items: int = 5):
    """Render an inline player + label for each recommended recording that's on disk."""
    for row in recommendations.head(max_items).itertuples():
        filename = getattr(row, "filename", None)
        if not filename:
            print(f"skip {row.recordingId}: no filename in metadata")
            continue

        audio_path = AUDIO_DIR / filename
        if not audio_path.exists():
            print(f"skip {filename}: not found in {AUDIO_DIR} (was it downloaded in notebook 1?)")
            continue

        score = getattr(row, "score", None)
        label = f"{filename}" + (f"  score={score:.3f}" if score is not None else "")
        snippet = getattr(row, "transcript_snippet", "")
        if snippet:
            label += f'  — "{snippet}..."'

        print(label)
        display(Audio(filename=str(audio_path)))


recommendations = recommend_for_user(USER_ID)
play_recommendations(recommendations)
